# 02 · Preparación y transformación de los datos (Fase 2)

**Proyecto:** comportamiento de compra en una tienda de cosméticos en línea
**Responsable de este notebook:** Gabriela (ingeniería de datos)
**Actividades de la guía que cubre:** 9 (estandarizar), 10 (integrar) y 14 (proceso reproducible)

Este notebook explica y ejecuta, paso a paso, el proceso que va de los datos
originales al dataset preparado. El trabajo real lo hacen los scripts de
`src/`; aquí se ejecutan en orden y se muestran sus verificaciones para que
cualquiera pueda revisarlo sin explicación oral.

| Paso | Script | Qué produce |
|---|---|---|
| 0 | `01_ingesta.py` | Parquet fiel a los CSV + mediciones de volumen |
| 1 | `02_muestra.py` | Muestra del 5 % de usuarios (semilla 42) |
| 2 | `03_limpieza.py` | Eventos estandarizados y con banderas de calidad |
| 3 | `04_integracion.py` | Tabla de eventos integrada + tabla de productos |
| 4 | `05_verificar_raw.py` | Comprobación de que los CSV originales no cambiaron |

**Cómo usarlo.** Por defecto trabaja con la **muestra** (`FUENTE = "muestra"`),
que tarda pocos segundos. Para el dataset final, cambie a `FUENTE = "completo"`
y vuelva a ejecutar todo (*Run All*). Los resultados que se ven guardados en
este archivo corresponden a la muestra.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

import polars as pl

FUENTE = "muestra"   # "muestra" o "completo"

# Ubicar la raíz del repositorio (la carpeta que contiene src/config.py),
# sin importar desde dónde se abra el notebook.
RAIZ = Path.cwd()
while not (RAIZ / "src" / "config.py").exists():
    if RAIZ.parent == RAIZ:
        raise FileNotFoundError("No se encontró la carpeta del proyecto")
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import config  # rutas y decisiones compartidas por todos los scripts

pl.Config(tbl_rows=30, tbl_cols=-1, tbl_width_chars=160, fmt_str_lengths=60,
          float_precision=2, thousands_separator=",")
TABLAS = config.TABLAS


def ejecutar(script, *argumentos):
    """Ejecuta un script de src/ y muestra lo que imprime."""
    entorno = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    r = subprocess.run([sys.executable, str(RAIZ / "src" / script), *argumentos],
                       capture_output=True, text=True, encoding="utf-8",
                       env=entorno, cwd=RAIZ)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} terminó con error")


print("Proyecto:", RAIZ.name)
print("Fuente:", FUENTE, "| Polars", pl.__version__)

Proyecto: proyecto-bigdata-cosmeticos
Fuente: muestra | Polars 2.1.0


## Paso 0 · Datos originales y conversión a Parquet

Los cinco CSV de Kaggle se guardan en `data/raw/` y **nunca se modifican**.
`01_ingesta.py` los copia a Parquet sin cambiar ningún valor; solo agrega dos
columnas de trazabilidad (`archivo_origen` y `fila_origen`) para poder volver
de cualquier fila procesada a su fila exacta en el CSV.

La tabla siguiente es la medición hecha en la computadora del equipo. Muestra
por qué se trabaja con Parquet y Polars: el Parquet ocupa mucho menos espacio
y las consultas son varias veces más rápidas que sobre el CSV.

In [2]:
mediciones = pl.read_csv(TABLAS / "01_mediciones_ingesta.csv")
mediciones.select(
    "archivo", "filas", "csv_mib", "parquet_mib", "reduccion_tamano_pct",
    "csv_agregacion_s", "parquet_agregacion_s",
    *[c for c in ("pandas_tiempo_carga_s", "pandas_memoria_gib") if c in mediciones.columns],
)

archivo,filas,csv_mib,parquet_mib,reduccion_tamano_pct,csv_agregacion_s,parquet_agregacion_s,pandas_tiempo_carga_s,pandas_memoria_gib
str,i64,f64,f64,f64,f64,f64,f64,f64
"""2019-Oct.csv""","4,102,283",460.20,75.20,83.70,0.30,0.06,4.83,1.23
"""2019-Nov.csv""","4,635,837",520.60,83.40,84.00,0.29,0.06,5.33,1.39
"""2019-Dec.csv""","3,533,286",396.10,67.10,83.10,0.28,0.05,4.45,1.06
"""2020-Jan.csv""","4,264,752",478.50,79.10,83.50,0.31,0.06,6.00,1.28
"""2020-Feb.csv""","4,156,682",466.20,76.40,83.60,0.29,0.06,5.08,1.25
"""TOTAL""","20,692,840","2,321.50",381.20,83.60,1.47,0.29,25.69,6.22


## Paso 1 · Muestra de trabajo

Todo el código se prueba primero con una muestra y después se ejecuta con los
datos completos. La muestra contiene al **5 % de los usuarios** con **todos**
sus eventos; se eligen usuarios y no filas, para no partir sesiones. La
selección usa SHA-256 sobre `"42:<user_id>"`, así que es idéntica en cualquier
computadora.

La comparación muestra que la muestra es representativa: los porcentajes de
cada tipo de evento, de faltantes y el precio casi no cambian.

In [3]:
pl.read_csv(TABLAS / "02_representatividad_muestra.csv").transpose(
    include_header=True, header_name="indicador", column_names="conjunto"
)

indicador,completo,muestra
str,f64,f64
"""eventos""","20,692,840.00","1,040,138.00"
"""usuarios""","1,639,358.00","81,767.00"
"""sesiones""","4,535,941.00","228,186.00"
"""productos""","54,571.00","41,371.00"
"""pct_view""",46.67,46.40
"""pct_cart""",27.88,28.20
"""pct_remove_from_cart""",19.23,19.43
"""pct_purchase""",6.22,5.97
"""pct_brand_nulo""",42.32,42.04


## Paso 2 · Estandarización y limpieza (actividad 9)

`03_limpieza.py` procesa cada mes y aplica estas reglas:

| Elemento | Antes | Después | Justificación |
|---|---|---|---|
| `event_time` | texto `"2019-10-01 00:00:04 UTC"` | fecha-hora con zona **UTC** | Permite operar con fechas. La fuente no indica el país, así que las horas son UTC, no hora local |
| Fecha derivada | — | `fecha`, `hora` (0–23), `dia_semana` (1 = lunes … 7 = domingo) | Las pide el análisis del embudo por día y hora (P1) |
| `category_code` | texto con puntos, 98 % vacío | 3 niveles (`categoria_nivel_1..3`); vacío → `"unknown"` | Si el código tiene solo dos niveles, el tercero queda vacío (no aplica) |
| Textos | `event_type`, `category_code`, `brand` | minúsculas y sin espacios sobrantes | Evita que una misma marca cuente como dos |
| Nombres de columnas | ya en snake_case | se conservan | Mantienen la trazabilidad con la fuente; las nuevas también van en snake_case |
| Tipos | `Int64`, `String` | `Enum`, `Int32`, `Categorical`, `Datetime` | Menos memoria |

**Decisiones de la actividad 8 (las define Rodrigo).** Por defecto **no se
elimina ninguna fila**: cada problema queda marcado con una columna bandera,
para poder medir el efecto de eliminarlo antes de decidir, como pide la guía.

| Problema | Bandera | Decisión aplicada por defecto |
|---|---|---|
| Duplicado exacto (las 9 columnas iguales) | `es_duplicado_exacto`, `repeticiones_exactas` | Se marca la 2.ª aparición y las siguientes; no se elimina |
| Precio 0 | `precio_cero` | Se marca |
| Precio negativo | `precio_negativo` | Se marca |
| Sesión nula | `sesion_nula` | Se marca; no se elimina |
| Marca nula | `marca_nula_original` | Se intenta recuperar en el paso 3 |
| Categoría textual nula | `categoria_code_nula` | `"unknown"` |

Las decisiones se cambian en un solo lugar (`src/config.py`):

In [4]:
config.DECISIONES

{'eliminar_duplicados_exactos': False,
 'eliminar_sesion_nula': False,
 'recuperar_marca_desde_producto': True}

In [5]:
ejecutar("03_limpieza.py", "--fuente", FUENTE)

03 - LIMPIEZA Y ESTANDARIZACIÓN (muestra)
Decisiones aplicadas: {'eliminar_duplicados_exactos': False, 'eliminar_sesion_nula': False, 'recuperar_marca_desde_producto': True}
  2019-Oct:    203,383 filas | duplicados marcados 11,033 | memoria (9 columnas) 19.7 -> 15.0 MiB
  2019-Nov:    229,430 filas | duplicados marcados 11,385 | memoria (9 columnas) 22.3 -> 16.9 MiB
  2019-Dec:    182,960 filas | duplicados marcados 9,419 | memoria (9 columnas) 17.7 -> 13.5 MiB
  2020-Jan:    208,853 filas | duplicados marcados 10,835 | memoria (9 columnas) 20.2 -> 15.4 MiB
  2020-Feb:    215,512 filas | duplicados marcados 12,891 | memoria (9 columnas) 20.9 -> 15.8 MiB

Filas: 1,040,138 -> 1,040,138 (eliminadas: 0)
Memoria de las 9 columnas originales: 100.8 -> 76.6 MiB (24 % menos)
Duración: 2.0 s
Salida: data/interim/limpio/muestra



### Efecto de la limpieza

Por cada mes: cuántas filas entraron y salieron, cuántas tienen cada problema
y cuánta memoria ocupan las 9 columnas originales antes y después del cambio
de tipos.

In [6]:
pl.read_csv(TABLAS / f"03_resumen_limpieza_{FUENTE}.csv")

archivo,filas_entrada,fechas_invalidas,textos_modificados_event_type,textos_modificados_category_code,textos_modificados_brand,duplicados_exactos_marcados,precio_cero,precio_negativo,sesion_nula,marca_nula,category_code_nulo,filas_eliminadas,filas_salida,memoria_originales_antes_mib,memoria_originales_despues_mib,memoria_total_salida_mib
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""2019-Oct""","203,383",0,0,0,0,"11,033",170,2,32,"83,965","200,122",0,"203,383",19.70,15.00,20.40
"""2019-Nov""","229,430",0,0,0,0,"11,385",474,1,55,"96,975","225,519",0,"229,430",22.30,16.90,23.00
"""2019-Dec""","182,960",0,0,0,0,"9,419",383,0,29,"77,523","179,940",0,"182,960",17.70,13.50,18.30
"""2020-Jan""","208,853",0,0,0,0,"10,835",379,3,45,"86,576","205,069",0,"208,853",20.20,15.40,20.90
"""2020-Feb""","215,512",0,0,0,0,"12,891","2,439",4,46,"92,259","211,680",0,"215,512",20.90,15.80,21.60
"""TOTAL""","1,040,138",0,0,0,0,"55,563","3,845",10,207,"437,298","1,022,330",0,"1,040,138",100.80,76.60,104.20


### Antes y después en un mismo evento

Mismo evento, leído del Parquet original y del limpio, gracias a
`archivo_origen` y `fila_origen`.

In [7]:
mes = config.MESES[0]
crudo = pl.read_parquet(config.carpeta_entrada(FUENTE) / f"{mes}.parquet")
limpio = pl.read_parquet(config.carpeta_limpio(FUENTE) / f"{mes}.parquet")

fila = limpio.filter(pl.col("categoria_code_nula").not_()).row(0, named=True)["fila_origen"]


def ficha(df):
    # Muestra una fila como tabla vertical: columna, tipo y valor.
    return pl.DataFrame({
        "columna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "valor": [str(v) for v in df.row(0)],
    })


print("ANTES (Parquet original)")
display(ficha(crudo.filter(pl.col("fila_origen") == fila)))
print("DESPUÉS (limpio)")
display(ficha(limpio.filter(pl.col("fila_origen") == fila)))

ANTES (Parquet original)


columna,tipo,valor
str,str,str
"""event_time""","""String""","""2019-10-01 00:07:58 UTC"""
"""event_type""","""String""","""remove_from_cart"""
"""product_id""","""Int64""","""5767493"""
"""category_id""","""Int64""","""1487580013053083824"""
"""category_code""","""String""","""stationery.cartrige"""
"""brand""","""String""","""italwax"""
"""price""","""Float64""","""2.14"""
"""user_id""","""Int64""","""514753614"""
"""user_session""","""String""","""e2fecb2d-22d0-df2c-c661-15da44b3ccf1"""


DESPUÉS (limpio)


columna,tipo,valor
str,str,str
"""event_time""","""Datetime(time_unit='us', time_zone='UTC')""","""2019-10-01 00:07:58+00:00"""
"""event_type""","""Enum(categories=['view', 'cart', 'remove_from_cart', 'purcha…","""remove_from_cart"""
"""product_id""","""Int32""","""5767493"""
"""category_id""","""Int64""","""1487580013053083824"""
"""category_code""","""Categorical""","""stationery.cartrige"""
"""brand""","""Categorical""","""italwax"""
"""price""","""Float64""","""2.14"""
"""user_id""","""Int32""","""514753614"""
"""user_session""","""String""","""e2fecb2d-22d0-df2c-c661-15da44b3ccf1"""


## Paso 3 · Integración (actividad 10)

`04_integracion.py` une los cinco meses y construye la tabla de productos. Las
reglas son explícitas y cada unión se verifica contando filas antes y después.

**Llaves**

| Tabla de eventos | Se relaciona con | Cardinalidad |
|---|---|---|
| `product_id` | `productos.product_id` | muchos a uno |
| `user_session` | sesión (las nulas se excluyen) | muchos a uno |
| `user_id` | usuario | muchos a uno |

**Reglas**

1. Los cinco meses deben tener las mismas columnas y tipos, cada archivo solo
   trae fechas de su mes y los rangos no se superponen.
2. **Tabla de productos (una fila por `product_id`)**: si un producto aparece
   con varias categorías o marcas, se usa **la más frecuente**. Si hay empate,
   la del evento más reciente y, si persiste, el valor menor.
3. **Recuperación de marca**: un evento sin marca toma la marca de su producto
   **solo si el producto tiene una única marca conocida**. Si el producto tiene
   varias, no se adivina. Lo que sigue sin marca queda como `"unknown"`.
4. **Sesiones**: se marcan las que tienen más de un `user_id`
   (`sesion_multiusuario`) y se cuentan las que cruzan de un mes a otro. Por
   eso las tablas de sesiones se deben construir con los cinco meses unidos.

Si alguna verificación falla, el script se detiene.

In [8]:
ejecutar("04_integracion.py", "--fuente", FUENTE)

04 - INTEGRACIÓN (muestra)

[1] Esquemas
  meses con columnas y tipos idénticos: True  [OK]

[2] Rangos de fechas
  eventos fuera del mes de su archivo: 0  [OK]
  pares de meses con fechas superpuestas: 0  [OK]

[3] Unión
  filas tras unir los 5 meses: 1040138  [OK]

[4] Tabla de productos
  filas en productos = product_id distintos: 41371  [OK]
  product_id único en productos: 0  [OK]
  producto_multicategoria: 1024
  producto_multimarca: 15
  producto_sin_marca: 17813
  producto_marca_recuperable: 1144
  producto_precio_negativo: 3

[5] Recuperación de marca
  filas tras unir con productos: 1040138  [OK]
  eventos con marca nula (original): 437298
  eventos con marca recuperada: 4315
  eventos que quedan con marca 'unknown': 432983  [OK]

[6] Sesiones
  sesiones distintas (sin nulas): 228186
  sesiones con más de un user_id: 0
  sesiones que cruzan de un mes a otro: 775

[7] Guardado
  filas guardadas en eventos.parquet: 1040138  [OK]

Duración: 0.9 s
Salida: data/processed/muestra
T

In [9]:
pl.read_csv(TABLAS / f"04_verificaciones_integracion_{FUENTE}.csv")

paso,valor,esperado,resultado,detalle
str,str,str,str,str
"""meses con columnas y tipos idénticos""","""True""","""True""","""OK""","""24 columnas"""
"""eventos fuera del mes de su archivo""","""0""","""0""","""OK""",""""""
"""pares de meses con fechas superpuestas""","""0""","""0""","""OK""",""""""
"""filas tras unir los 5 meses""","""1040138""","""1040138""","""OK""","""debe ser la suma de los meses"""
"""filas en productos = product_id distintos""","""41371""","""41371""","""OK""",""""""
"""product_id único en productos""","""0""","""0""","""OK""",""""""
"""producto_multicategoria""","""1024""","""""","""""",""""""
"""producto_multimarca""","""15""","""""","""""",""""""
"""producto_sin_marca""","""17813""","""""","""""",""""""


### Ejemplos de las reglas aplicadas

In [10]:
salida = config.carpeta_procesado(FUENTE)
productos = pl.read_parquet(salida / "productos.parquet")
eventos = pl.scan_parquet(salida / "eventos.parquet")

print("Producto con dos categorías: se elige la más frecuente")
pid = productos.filter("producto_multicategoria")["product_id"][0]
display(eventos.filter(pl.col("product_id") == pid)
        .group_by("category_id").agg(pl.len().alias("eventos")).sort("eventos", descending=True).collect())
display(productos.filter(pl.col("product_id") == pid)
        .select("product_id", "categoria_principal", "categorias_distintas"))

print("Productos que solo aparecen con precio negativo (probables descuentos o cupones)")
display(productos.filter("producto_precio_negativo")
        .select("product_id", "categoria_principal", "precio_max", "eventos"))

Producto con dos categorías: se elige la más frecuente


category_id,eventos
i64,u32
"2,196,456,817,758,831,535",10
"1,487,580,006,174,425,994",8


product_id,categoria_principal,categorias_distintas
i32,i64,u32
"4,382","2,196,456,817,758,831,535",2


Productos que solo aparecen con precio negativo (probables descuentos o cupones)


product_id,categoria_principal,precio_max,eventos
i32,i64,f64,u32
"5,670,257","1,487,580,014,042,939,619",-15.87,5
"5,716,855","1,487,580,014,042,939,619",-7.94,2
"5,716,859","1,487,580,014,042,939,619",-47.62,3


In [11]:
print("Recuperación de marca: un producto con una sola marca conocida")
pid = productos.filter("producto_marca_recuperable").sort("eventos_sin_marca", descending=True)["product_id"][0]
eventos.filter(pl.col("product_id") == pid).group_by(
    "marca_nula_original", "marca_recuperada", "brand"
).agg(pl.len().alias("eventos")).collect()

Recuperación de marca: un producto con una sola marca conocida


marca_nula_original,marca_recuperada,brand,eventos
bool,bool,cat,u32
true,true,"""estel""",140
false,false,"""estel""",129


## Paso 4 · Proceso reproducible (actividad 14)

* Los CSV de `data/raw/` no se modifican: `05_verificar_raw.py` recalcula su
  huella SHA-256 y la compara con la registrada en la ingesta.
* Todo el proceso se ejecuta con un solo comando, sin pasos manuales:

```
python src/run_pipeline.py                  # desde cero, datos completos
python src/run_pipeline.py --fuente muestra # prueba rápida con la muestra
```

* Las versiones de las librerías están en `requirements.txt`.

In [12]:
if all((config.RAW / f"{m}.csv").exists() for m in config.MESES):
    ejecutar("05_verificar_raw.py")
else:
    print("Los CSV originales no están en data/raw/ en este equipo; "
          "la verificación se ejecuta donde están los datos completos.")

Los CSV originales no están en data/raw/ en este equipo; la verificación se ejecuta donde están los datos completos.


## Resultado · Dataset preparado

| Archivo | Una fila por | Contenido |
|---|---|---|
| `data/processed/<fuente>/eventos.parquet` | evento | 9 columnas originales estandarizadas, fecha derivada, niveles de categoría, banderas de calidad y trazabilidad |
| `data/processed/<fuente>/productos.parquet` | producto | categoría y marca principales, precios, fechas y banderas del producto |

Las tablas de sesiones y usuarios (actividad 11) y la partición
entrenamiento / validación / prueba (actividad 13) se construyen a partir de
`eventos.parquet`.

**Filtro sugerido para análisis** mientras Rodrigo define la actividad 8:
excluir duplicados exactos y precios no positivos al calcular montos.

In [13]:
print("Columnas de eventos:")
for nombre, tipo in eventos.collect_schema().items():
    print(f"  {nombre:24s} {tipo}")

tamanos = {f.name: f"{f.stat().st_size / 1024**2:.1f} MiB" for f in salida.glob("*.parquet")}
print("\nArchivos:", tamanos)

eventos.filter(
    pl.col("es_duplicado_exacto").not_() & (pl.col("price") > 0)
).group_by("event_type").agg(pl.len().alias("eventos")).sort("event_type").collect()

Columnas de eventos:
  event_time               Datetime(time_unit='us', time_zone='UTC')
  event_type               Enum(categories=['view', 'cart', 'remove_from_cart', 'purchase'])
  product_id               Int32
  category_id              Int64
  category_code            Categorical
  brand                    Categorical
  price                    Float64
  user_id                  Int32
  user_session             String
  fecha                    Date
  hora                     Int8
  dia_semana               Int8
  categoria_nivel_1        Categorical
  categoria_nivel_2        Categorical
  categoria_nivel_3        Categorical
  es_duplicado_exacto      Boolean
  repeticiones_exactas     UInt32
  precio_cero              Boolean
  precio_negativo          Boolean
  sesion_nula              Boolean
  marca_nula_original      Boolean
  categoria_code_nula      Boolean
  archivo_origen           Enum(categories=['2019-Oct', '2019-Nov', '2019-Dec', '2020-Jan', '2020-Feb'])
  fila_or

event_type,eventos
enum,u32
"""view""","481,375"
"""cart""","285,955"
"""remove_from_cart""","151,923"
"""purchase""","62,000"
